In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, fisher_exact, describe
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

today = date.today()

In [2]:
def kde_wrapper(data, bw=0.1):
    fig = plt.figure()
    fig.patch.set_facecolor('white')
    
    sns.kdeplot(data, bw_adjust=bw)
    print(describe(data))
    return fig

In [3]:
f = open('/home/ctrader/region_similarity/data/regions_25kb_CT20220712.cleanedgenesmapped.bed', 'r')
mapped_genes = f.readlines()
f.close()

In [4]:
#loading order of each component
f = open('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.TFmapped.bed', 'r')
TF_component_order = np.array(f.readline().strip().split('\t'))[3:]
f.close()

f = open('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.DHSmapped.bed', 'r')
DHS_component_order = np.array(f.readline().strip().split('\t'))[3:]
f.close()

f = open('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.epilogosmapped.bed', 'r')
epilogos_component_order = np.array(f.readline().strip().split('\t'))[3:]
f.close()

In [5]:
#descriptors for each factor
pickle_in = open('/home/ctrader/region_similarity/data/tf_motif_descriptor_dict_CT20220627.pickle', 'rb')
TFfact2descr = pickle.load(pickle_in)

pickle_in = open('/home/ctrader/region_similarity/data/DHS_descriptor_dict_CT20220729.pickle', 'rb')
DHSfact2descr = pickle.load(pickle_in)

pickle_in = open('/home/ctrader/region_similarity/data/epilogos_descriptor_dict_CT20220627.pickle', 'rb')
epilogosfact2descr = pickle.load(pickle_in)

In [6]:
DHS_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.DHScosine.hdf5'
TF_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.TFcosine.hdf5'
epilogos_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.epilogoscosine.hdf5'
coex_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.coexpression_human_transferred_maxSTRINGsimilarity.hdf5'
# coextrans_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.coexpression_transferredSTRINGsimilarity.hdf5'
cooccur_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.cooccurenceSTRINGsimilarity.hdf5'
database_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.database_human_transferred_maxSTRINGsimilarity.hdf5'
exper_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.experiments_human_transferred_maxSTRINGsimilarity.hdf5'
# expertrans_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.experiments_transferredSTRINGsimilarity.hdf5'
fusion_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.fusionSTRINGsimilarity.hdf5'
homology_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.homologySTRINGsimilarity.hdf5'
neigh_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.neighborhood_human_transferred_maxSTRINGsimilarity.hdf5'
textmin_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220712.textmining_human_transferred_maxSTRINGsimilarity.hdf5'
# textmintrans_hdf5_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.textmining_transferredSTRINGsimilarity.hdf5'

DHS_ooe_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.DHSoee.hdf5'
TF_ooe_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.TFoee.hdf5'
epilogos_ooe_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.epilogosoee.hdf5'

DHS_mapped_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.DHSmapped.bed'
TF_mapped_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.TFmapped.bed'
epilogos_mapped_path = '/home/ctrader/region_similarity/data/regions_25kb_CT20220708.epilogosmapped.bed'

regions_path = '/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220613.sorted.c1t3.bed'
gz_file_path = '/home/ctrader/region_similarity/data/Altius_25kb_hg38_maxMean_CT20220613.sorted.c1t3.rownum4.bed.gz'

geneID2name_path = '/home/ctrader/region_similarity/data/gene_proteinID2genename_dict_CT20220712.pickle'

In [169]:
DHS_ooe_hdf5 = h5py.File('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.DHSoee.hdf5', 'r')
DHS_ooe = DHS_ooe_hdf5['ooe_arr'][:]
DHS_ooe_hdf5.close()
TF_ooe_hdf5 = h5py.File('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.TFoee.hdf5', 'r')
TF_ooe = TF_ooe_hdf5['ooe_arr'][:]
TF_ooe_hdf5.close()
epilogos_ooe_hdf5 = h5py.File('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.epilogosoee.hdf5', 'r')
epilogos_ooe = epilogos_ooe_hdf5['ooe_arr'][:]
epilogos_ooe_hdf5.close()

In [7]:
pickle_in = open('/home/ctrader/region_similarity/data/gene_proteinID2genename_dict_CT20220712.pickle', 'rb')
geneID2name = pickle.load(pickle_in)

In [131]:
def combine_scores(DHS_count_scaled_vals, DHS_count_similar_regions, TF_count_scaled_vals, TF_count_similar_regions, epilogos_score_scaled_vals,
                   epilogos_score_similar_regions, coex_scaled_vals, coex_similar_regions, cooccur_scaled_vals, cooccur_similar_regions,
                   database_scaled_vals, database_similar_regions, exper_scaled_vals, exper_similar_regions,fusion_scaled_vals, fusion_similar_regions,
                   homology_scaled_vals, homology_similar_regions, neigh_scaled_vals, neigh_similar_regions, textmin_scaled_vals, textmin_similar_regions):
    #max ind for combined score array
    max_ind = np.max([DHS_count_similar_regions, TF_count_similar_regions, epilogos_score_similar_regions, coex_similar_regions,
                     cooccur_similar_regions, database_similar_regions, exper_similar_regions, fusion_similar_regions, homology_similar_regions,
                     neigh_similar_regions, textmin_similar_regions])

    #DHS scores
    DHS_arr = create_arr(DHS_count_similar_regions, DHS_count_scaled_vals, max_ind)
    #epilogos scores
    epilogos_arr = create_arr(epilogos_score_similar_regions, epilogos_score_scaled_vals, max_ind)
    #TF motif scores
    TF_arr = create_arr(TF_count_similar_regions, TF_count_scaled_vals, max_ind)
    #coexpression scores
    coex_arr = create_arr(coex_similar_regions, coex_scaled_vals, max_ind)
    #cooccurrence scores
    cooccur_arr = create_arr(cooccur_similar_regions, cooccur_scaled_vals, max_ind)
    #database scores
    database_arr = create_arr(database_similar_regions, database_scaled_vals, max_ind)
    #experiment scores
    exper_arr = create_arr(exper_similar_regions, exper_scaled_vals, max_ind)
    #fusion scores
    fusion_arr = create_arr(fusion_similar_regions, fusion_scaled_vals, max_ind)
    #homology scores
    homology_arr = create_arr(homology_similar_regions, homology_scaled_vals, max_ind)
    #neighborhood scores
    neigh_arr = create_arr(neigh_similar_regions, neigh_scaled_vals, max_ind)
    #text mining scores
    textmin_arr = create_arr(textmin_similar_regions, textmin_scaled_vals, max_ind)

    #scoring all metrics
    score_arr = 1 - ((1 - epilogos_arr) * (1 - DHS_arr) * (1 - TF_arr) * (1 - coex_arr) * (1 - cooccur_arr) * (1 - database_arr) * (1 - exper_arr) *
                     (1 - fusion_arr) * (1 - homology_arr) * (1 - neigh_arr) * (1 - textmin_arr))

    #labeling most contributing metrics for each region
    contributing_cosine_arr = np.argmax(np.array((DHS_arr, TF_arr, epilogos_arr, coex_arr, cooccur_arr, database_arr, exper_arr, fusion_arr,
                                                  homology_arr, neigh_arr, textmin_arr)), axis=0)

    return (score_arr, contributing_cosine_arr, max_ind, DHS_arr, TF_arr, epilogos_arr, coex_arr, 
            cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, neigh_arr, textmin_arr)

def region_return(score_arr, contributing_cosine_arr, roi, regions_path, DHS_ooe_path, TF_ooe_path, epilogos_ooe_path, DHS_mapped_path, TF_mapped_path,
                  epilogos_mapped_path, coex_genes, cooccur_genes, database_genes, exper_genes, fusion_genes, homology_genes, neigh_genes,
                  textmin_genes, geneID2name_path):

    #top 100 region inds and scores
    rec_inds = np.argsort(-score_arr)[:, :100]
    top_region_inds = np.hstack((roi.reshape(-1, 1), rec_inds))
    # top_region_scores = np.append(1, score_arr[rec_inds])

    return top_region_inds

def roi_url(roi, top_regions):
    today = date.today()
    roi_url_name = 'region_comparison_recommended_regions_roi_'+str(roi)+'_scale_25kb_CT' + today.strftime("%Y%m%d") + '.bed'
    bed_path_name = os.path.join('/home/ctrader/public_html/', roi_url_name)
    np.savetxt(bed_path_name, top_regions, delimiter='\t', fmt='%s')
    link_path1 = 'https://epilogos.altius.org:3001/?application=viewer&sampleSet=vC&mode=single&genome=hg38&model=18&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + roi_url_name
    link_path2 = 'https://index.altius.org:3000/?application=viewer&mode=pd&chrLeft=chr11&chrRight=chr11&start=47329800&stop=47356800&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + roi_url_name
    return link_path1, link_path2, bed_path_name

def load_hdf5_files(hdf5_path, roi):
    #take in a file path and load scaled scores and similar regions
    hdf5_file = h5py.File(hdf5_path, 'r')
    scaled_vals = hdf5_file['top_1000_scaled_vals'][roi]
    similar_regions = hdf5_file['top_1000_similar_regions'][roi]
    hdf5_file.close()
    return scaled_vals, similar_regions

def load_hdf5_genes(hdf5_path, roi, similar_regions, max_ind):
    #load in tracking genes
    hdf5_file = h5py.File(hdf5_path, 'r')
    ori_genes = np.char.decode(hdf5_file['top_1000_original_genes'][roi], 'utf-8')
    dest_genes = np.char.decode(hdf5_file['top_1000_destination_genes'][roi], 'utf-8')
    #establishing gene arrays
    ori_arr = np.where(np.zeros((len(similar_regions), max_ind + 1))==0, None, None)
    dest_arr = np.where(np.zeros((len(similar_regions), max_ind + 1))==0, None, None)

    #filling gene arrays
    #find the values to change
    cols = similar_regions
    rows = np.array([np.arange(len(similar_regions))]*len(similar_regions[0])).T

    #set values
    ori_arr[rows, cols] = ori_genes
    dest_arr[rows, cols] = dest_genes
    return np.array([ori_arr, dest_arr])

def create_arr(similar_regions, scaled_vals, max_ind):
    #empty array
    arr = np.zeros((len(similar_regions), max_ind + 1))
    
    #find the values to change
    cols = similar_regions
    rows = np.array([np.arange(len(similar_regions))]*len(similar_regions[0])).T
    
    #set values
    arr[rows, cols] = scaled_vals
    return arr

In [260]:
roi = np.arange(5445-50, 5445+50)

In [262]:
print('roi = ' + str(roi), flush=True)
t = time.time()

###Load rank and ind arrays
#DHS counts
DHS_count_scaled_vals, DHS_count_similar_regions = load_hdf5_files(DHS_hdf5_path, roi)
#TF motif counts
TF_count_scaled_vals, TF_count_similar_regions = load_hdf5_files(TF_hdf5_path, roi)
#epilogos score sum and max
epilogos_score_scaled_vals, epilogos_score_similar_regions = load_hdf5_files(epilogos_hdf5_path, roi)
#coexpression STRING scores
coex_scaled_vals, coex_similar_regions = load_hdf5_files(coex_hdf5_path, roi)
#cooccurence STRING scores
cooccur_scaled_vals, cooccur_similar_regions = load_hdf5_files(cooccur_hdf5_path, roi)
#database STRING scores
database_scaled_vals, database_similar_regions = load_hdf5_files(database_hdf5_path, roi)
#experiments STRING scores
exper_scaled_vals, exper_similar_regions = load_hdf5_files(exper_hdf5_path, roi)
#fusion STRING scores
fusion_scaled_vals, fusion_similar_regions = load_hdf5_files(fusion_hdf5_path, roi)
#homology STRING scores
homology_scaled_vals, homology_similar_regions = load_hdf5_files(homology_hdf5_path, roi)
#neighborhood STRING scores
neigh_scaled_vals, neigh_similar_regions = load_hdf5_files(neigh_hdf5_path, roi)
#textmining STRING scores
textmin_scaled_vals, textmin_similar_regions = load_hdf5_files(textmin_hdf5_path, roi)

###Combine scores 
(score_arr, contributing_cosine_arr, max_ind, DHS_arr, TF_arr, epilogos_arr, coex_arr, 
 cooccur_arr, database_arr, exper_arr, fusion_arr, homology_arr, neigh_arr, textmin_arr) = combine_scores(DHS_count_scaled_vals, DHS_count_similar_regions, TF_count_scaled_vals, TF_count_similar_regions, epilogos_score_scaled_vals,
                                                                                                         epilogos_score_similar_regions, coex_scaled_vals, coex_similar_regions, cooccur_scaled_vals, cooccur_similar_regions,
                                                                                                         database_scaled_vals, database_similar_regions, exper_scaled_vals, exper_similar_regions, fusion_scaled_vals, fusion_similar_regions,
                                                                                                         homology_scaled_vals, homology_similar_regions, neigh_scaled_vals, neigh_similar_regions, textmin_scaled_vals, textmin_similar_regions)

#load in gene names for STRING scores
coex_genes = load_hdf5_genes(coex_hdf5_path, roi, coex_similar_regions, max_ind)
cooccur_genes = load_hdf5_genes(cooccur_hdf5_path, roi, cooccur_similar_regions, max_ind)
database_genes = load_hdf5_genes(database_hdf5_path, roi, database_similar_regions, max_ind)
exper_genes = load_hdf5_genes(exper_hdf5_path, roi, exper_similar_regions, max_ind)
fusion_genes = load_hdf5_genes(fusion_hdf5_path, roi, fusion_similar_regions, max_ind)
homology_genes = load_hdf5_genes(homology_hdf5_path, roi, homology_similar_regions, max_ind)
neigh_genes = load_hdf5_genes(neigh_hdf5_path, roi, neigh_similar_regions, max_ind)
textmin_genes = load_hdf5_genes(textmin_hdf5_path, roi, textmin_similar_regions, max_ind)

###Return regions
top_region_inds = region_return(score_arr, contributing_cosine_arr, roi, regions_path, DHS_ooe_path, TF_ooe_path, epilogos_ooe_path, DHS_mapped_path, TF_mapped_path,
                                               epilogos_mapped_path, coex_genes, cooccur_genes, database_genes, exper_genes, fusion_genes, homology_genes, neigh_genes,
                                               textmin_genes, geneID2name_path)

print(time.time()-t)

roi = [5395 5396 5397 5398 5399 5400 5401 5402 5403 5404 5405 5406 5407 5408
 5409 5410 5411 5412 5413 5414 5415 5416 5417 5418 5419 5420 5421 5422
 5423 5424 5425 5426 5427 5428 5429 5430 5431 5432 5433 5434 5435 5436
 5437 5438 5439 5440 5441 5442 5443 5444 5445 5446 5447 5448 5449 5450
 5451 5452 5453 5454 5455 5456 5457 5458 5459 5460 5461 5462 5463 5464
 5465 5466 5467 5468 5469 5470 5471 5472 5473 5474 5475 5476 5477 5478
 5479 5480 5481 5482 5483 5484 5485 5486 5487 5488 5489 5490 5491 5492
 5493 5494]
49.499335050582886


In [263]:
#find the values
cols = top_region_inds[:, 1:]
rows = np.array([np.arange(len(top_region_inds[:, 1:]))]*len(top_region_inds[:, 1:][0])).T

DHS_arr_top_100 = DHS_arr[rows, cols]
TF_arr_top_100 = TF_arr[rows, cols]
epilogos_arr_top_100 = epilogos_arr[rows, cols]

all_metrics_arr = np.array((DHS_arr, TF_arr, epilogos_arr, coex_arr, cooccur_arr, 
                             database_arr, exper_arr, fusion_arr, homology_arr, 
                             neigh_arr, textmin_arr))
string_arr_top_100 = all_metrics_arr[3:, rows, cols]

string_comp_order = ['Co-expressed with', 'Co-occurs within Species with', 'Co-annotated with', 
                     'Interacts in vitro with', 'Fused in Other Species with', 'Has Common Ancestory with', 
                     'is Nearby', 'Co-occurs in Literature with']
colors = np.array(['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c', '#fdbf6f', '#ff7f00', 
                   '#cab2d6', '#6a3d9a', '#ffff99'])

#roi comps
DHS_roi_rank = rankdata(-DHS_ooe[top_region_inds[:, 0]], axis=1)
TF_roi_rank = rankdata(-TF_ooe[top_region_inds[:, 0]], axis=1)
epilogos_roi_mean_rank = rankdata(-epilogos_ooe[top_region_inds[:, 0]][:, :18], axis=1)
epilogos_roi_max_rank = rankdata(-epilogos_ooe[top_region_inds[:, 0]][:, 18:], axis=1)




In [268]:
#find genomic locations
f = open(regions_path, 'r')
regions = f.readlines()
regions_for_sentences = []
for ind in roi:
    regions_for_sentences.append(regions[ind].strip().split())
f.close()

In [279]:
file = open('/home/ctrader/region_similarity/data/sentence_summaries/regions_25kb_5395_through_5494_sentence_summaries_CT20220803.txt', 'w')

for r in range(len(roi)):
    print(roi[r])
    color_tracker = np.array([])
    topfactor2region = {}
    #find top comps for each metric/recommended region
    for i in range(len(DHS_arr_top_100[r])):
        if DHS_arr_top_100[r][i] > 0:
            DHS_i_rank = rankdata(-DHS_ooe[top_region_inds[r][i+1]])
            top_comp = np.argsort((DHS_roi_rank[r]+DHS_i_rank)/2)[0]
            f = DHSfact2descr[DHS_component_order[top_comp]]
            if f not in topfactor2region:
                topfactor2region[f] = np.array([i])
                color_tracker = np.append(color_tracker, colors[0])
            else:
                topfactor2region[f] = np.append(topfactor2region[f], [i])
                
    for i in range(len(TF_arr_top_100[r])):
        if TF_arr_top_100[r][i] > 0:
            TF_i_rank = rankdata(-TF_ooe[top_region_inds[r][i+1]])
            top_comp = np.argsort((TF_roi_rank[r]+TF_i_rank)/2)[0]
            f = TF_component_order[top_comp] + ': ' + TFfact2descr[TF_component_order[top_comp]]
            if f not in topfactor2region:
                topfactor2region[f] = np.array([i])
                color_tracker = np.append(color_tracker, colors[1])
            else:
                topfactor2region[f] = np.append(topfactor2region[f], [i])

    for i in range(len(epilogos_arr_top_100[r])):
        if epilogos_arr_top_100[r][i] > 0:
            epilogos_i_mean_rank = rankdata(-epilogos_ooe[top_region_inds[r][i+1]][:18])
            epilogos_i_max_rank = rankdata(-epilogos_ooe[top_region_inds[r][i+1]][18:])
            top_mean_comp = np.argsort((epilogos_roi_mean_rank[r] + epilogos_i_mean_rank)/2)[0]
            top_max_comp = np.argsort((epilogos_roi_max_rank[r] + epilogos_i_max_rank)/2)[0]
            if top_mean_comp==top_max_comp:
                f = epilogosfact2descr[epilogos_component_order[top_mean_comp]]
            else:
                f = (epilogosfact2descr[epilogos_component_order[top_mean_comp]] + ', with ' + 
                     epilogosfact2descr[epilogos_component_order[top_max_comp]] + ' peaks')
            if f not in topfactor2region:
                topfactor2region[f] = np.array([i])
                color_tracker = np.append(color_tracker, colors[2])
            else:
                topfactor2region[f] = np.append(topfactor2region[f], [i])

    #all string original and destination genes
    all_arr_ori = np.array([coex_genes[0, r, :], cooccur_genes[0, r, :], database_genes[0, r, :], exper_genes[0, r, :], 
                            fusion_genes[0, r, :], homology_genes[0, r, :], neigh_genes[0, r, :], textmin_genes[0, r, :]])[:, top_region_inds[r, 1:]]
    all_arr_dest = np.array([coex_genes[1, r, :], cooccur_genes[1, r, :], database_genes[1, r, :], exper_genes[1, r, :], 
                             fusion_genes[1, r, :], homology_genes[1, r, :], neigh_genes[1, r, :], textmin_genes[1, r, :]])[:, top_region_inds[r, 1:]]

    #STRING Scores
    for i in range(len(string_arr_top_100[0, r, :])):
        if np.sum(string_arr_top_100[:, r, i]) > 0:
            pos_string_inds = np.where(string_arr_top_100[:, r, i]>0)[0]
            for comp in pos_string_inds:
                ori_gene = geneID2name[all_arr_ori[comp, i]]
                f = 'Gene ' + string_comp_order[comp] + ' ' + ori_gene
                if f not in topfactor2region:
                    topfactor2region[f] = np.array([i])
                    color_tracker = np.append(color_tracker, colors[comp+3])
                else:
                    topfactor2region[f] = np.append(topfactor2region[f], [i])
                    

    upset_data = upsetplot.from_contents(topfactor2region)
    upset_data = upset_data.groupby(list(upset_data.index.names)).count()['id']

    #sorting upset categories
    upset_factors = np.array(upset_data.index.names)
    upset_factor_counts = np.array([])
    for i in range(len(upset_factors)):
        upset_factor_counts = np.append(upset_factor_counts, np.sum(upset_data.unstack(level=i)[True]))
    upset_factors_sorted = upset_factors[np.argsort(upset_factor_counts)]
    sorted_upset_data = upset_data.reorder_levels(upset_factors_sorted)
    color_tracker_sorted = color_tracker[np.argsort(upset_factor_counts)]


    ######
    limiting_per = 95

    overall_set = set()
    topfactor2region_limited = {}
    ind = 0
    while len(overall_set) < limiting_per:
        factor = np.flip(list(sorted_upset_data.index.names))[ind]
        factor_set = set(list(topfactor2region[factor]))
        overall_set.update(factor_set)
        ind += 1
        topfactor2region_limited[factor] = topfactor2region[factor]

    upset_data_limited = upsetplot.from_contents(topfactor2region_limited)
    upset_data_limited = upset_data_limited.groupby(list(upset_data_limited.index.names)).count()['id']

    #sorting upset categories
    upset_factors_limited = np.array(upset_data_limited.index.names)
    upset_factor_counts_limited = np.array([])
    if len(upset_data_limited) > 1:
        for i in range(len(upset_factors_limited)):
            upset_factor_counts_limited = np.append(upset_factor_counts_limited, np.sum(upset_data_limited.unstack(level=i)[True]))
        upset_factors_sorted_limited = upset_factors_limited[np.argsort(upset_factor_counts_limited)]
        sorted_upset_data_limited = upset_data_limited.reorder_levels(upset_factors_sorted_limited)
    else:
        upset_factors_sorted_limited = [upset_data_limited.index.name]

    ####
    #write a sentence based on 95% plot
    sentence = 'Recommended regions are similar to the query region due to '
    last_factor = False
    for factor in upset_factors_sorted_limited[::-1]:
        if factor == upset_factors_sorted_limited[0]:
            last_factor = True
        if (last_factor) & (len(upset_factors_sorted_limited) > 1):
            sentence += 'and '

        #each type of factor
        factor = factor.split()
        if factor[0] == 'Gene': #STRING
            factor[0] = 'a gene that'
            factor[1] = factor[1]
            sentence += ' '.join(factor[:-1]).lower() + ' '+ factor[-1]
        elif factor[0] == 'DNase': #DHSs
            sentence += ' '.join(factor)
        else: #TFs
            sentence += ' '.join(factor[1:]) + ' transcription factor occurrence'

        #seperate factors
        if last_factor:
            sentence += '.'
        else:
            sentence += ', '

    print(sentence)
    file.write(regions_for_sentences[r][0] + '\t' + regions_for_sentences[r][1] + '\t' + 
               regions_for_sentences[r][2] + '\t' + str(roi[r]) + '\t' + sentence + '\n')
    
file.close()

5395
Recommended regions are similar to the query region due to a gene that co-expressed with F13B, a gene that co-occurs in literature with F13B, DNase I accessibility in Vascular/endothelial samples, and DNase I accessibility in Cancer/epithelial samples.
5396
Recommended regions are similar to the query region due to a gene that co-expressed with F13B, a gene that co-occurs in literature with F13B, DNase I accessibility in Digestive samples, and  transcription factor occurrence.
5397
Recommended regions are similar to the query region due to a gene that co-expressed with ASPM.
5398
Recommended regions are similar to the query region due to a gene that co-expressed with ASPM.
5399
Recommended regions are similar to the query region due to a gene that co-expressed with ASPM.
5400
Recommended regions are similar to the query region due to DNase I accessibility in Cardiac samples, a gene that co-occurs in literature with ZBTB41, and a gene that co-expressed with ZBTB41.
5401
Recommended

Recommended regions are similar to the query region due to a gene that co-occurs in literature with NEK7, and DNase I accessibility in Cardiac samples.
5435
Recommended regions are similar to the query region due to a gene that co-occurs in literature with NEK7, a gene that co-expressed with NEK7, a gene that has common ancestory with NEK7, a gene that interacts in vitro with NEK7, DNase I accessibility in Myeloid/erythroid samples, a gene that co-occurs within species with NEK7, and DNase I accessibility in Cancer/epithelial samples.
5436
Recommended regions are similar to the query region due to a gene that co-occurs in literature with NEK7, a gene that co-expressed with NEK7, a gene that has common ancestory with NEK7, a gene that interacts in vitro with NEK7, and DNase I accessibility in Vascular/endothelial samples.
5437
Recommended regions are similar to the query region due to  transcription factor occurrence, and DNase I accessibility in Lymphoid samples.
5438
Recommended regio

Recommended regions are similar to the query region due to DNase I accessibility in StromalA samples.
5471
Recommended regions are similar to the query region due to DNase I accessibility in StromalA samples, and DNase I accessibility in Organdevel/renal samples.
5472
Recommended regions are similar to the query region due to DNase I accessibility in Neural samples, genes + repeats transcription factor occurrence, DNase I accessibility in StromalB samples, chromodomain helicase DNA binding protein transcription factor occurrence, and DNase I accessibility in Placental/trophoblast samples.
5473
Recommended regions are similar to the query region due to  transcription factor occurrence, and DNase I accessibility in Tissueinvariant samples.
5474
Recommended regions are similar to the query region due to  transcription factor occurrence, and DNase I accessibility in Vascular/endothelial samples.
5475
Recommended regions are similar to the query region due to DNase I accessibility in Renal/